# Certificación Profesional en Desarrollador de Agentes de IA
## Clase 1 — Fundamentos prácticos de aplicaciones con LLMs

ITBA Educación Ejecutiva

<img src="https://upload.wikimedia.org/wikipedia/commons/e/e4/Artificial_neural_network.svg" width="320"/>

<sub>Fuente: Cburnett, Wikimedia Commons, CC BY-SA 3.0</sub>

## Objetivo de la clase

Comprender **qué es** un modelo de lenguaje, **qué puede y no puede hacer**, y con qué criterios se elige uno para construir una aplicación real.

Al final de la clase vas a poder:
- Explicar cómo genera texto un LLM y por qué es probabilístico.
- Diferenciar generación, embeddings y reranking.
- Armar una matriz de selección de modelos para un caso de uso dado.

## Agenda

1. Qué es un modelo de lenguaje
2. De RNN a Transformers (intuición + ecuaciones + arquitectura)
3. Tokens, ventana de contexto, BPE y generación probabilística
4. Generación vs. embeddings vs. reranking
5. Modelos propietarios vs. abiertos — costo, latencia, privacidad, specs reales
6. Presentación del Trabajo Práctico Final Integrador
7. Actividad: matriz de selección de modelos

## ¿Qué es un modelo de lenguaje?

Un LLM es una función que, dado un texto de entrada, **predice la distribución de probabilidad del próximo token**.

- No "sabe" hechos: aprendió patrones estadísticos del lenguaje.
- No razona como una base de datos: genera texto plausible, no necesariamente verdadero.
- Es la pieza base de una aplicación, no la aplicación en sí misma.

## Capacidades y limitaciones

**Capacidades**
- Comprensión y generación de lenguaje natural en múltiples dominios.
- Seguir instrucciones (system/user prompts).
- Generar código, resúmenes, clasificaciones, salidas estructuradas.

**Limitaciones**
- Alucinaciones: puede generar información falsa con total confianza.
- Conocimiento limitado a su corte de entrenamiento (salvo herramientas externas).
- Ventana de contexto finita.
- Sensible a cómo se formula el prompt.

## De RNN a Transformers (intuición)

- **RNNs**: procesan texto secuencialmente, palabra por palabra → difícil paralelizar, pierden contexto lejano.
- **Transformers (2017, "Attention is all you need")**: procesan toda la secuencia en paralelo, usando **self-attention** para que cada token "mire" a todos los demás y pondere su relevancia.
- Esto habilitó modelos mucho más grandes, entrenados en paralelo, con mejor manejo de contexto largo.

<img src="https://upload.wikimedia.org/wikipedia/commons/b/b5/Recurrent_neural_network_unfold.svg" width="420"/>

<sub>Fuente: fdeloche, Wikimedia Commons, CC BY-SA 4.0</sub>

## Arquitectura Transformer (referencia)

<img src="https://upload.wikimedia.org/wikipedia/commons/3/34/Transformer%2C_full_architecture.png" width="380"/>

<sub>Fuente: dvgodoy (dl-visuals), Wikimedia Commons, CC BY 4.0</sub>

No hace falta memorizar el diagrama — la idea clave es: encoder/decoder apilados, cada uno con capas de self-attention y una red feed-forward.

## Arquitectura Transformer — por bloques

1. **Embedding + positional encoding**: cada token → vector, más una codificación de posición (el Transformer no procesa en orden, necesita saber "dónde" está cada token).
2. **Encoder** (N capas apiladas): multi-head self-attention + feed-forward + residuales/layer norm.
3. **Decoder** (N capas apiladas): self-attention **enmascarada** (solo ve tokens anteriores) + cross-attention al encoder + feed-forward.
4. **Capa de salida**: proyección lineal + softmax sobre el vocabulario → distribución de probabilidad del próximo token.

Los LLMs conversacionales actuales (incluido Claude) son mayormente **decoder-only**: sin encoder separado, solo self-attention enmascarada + feed-forward, entrenados para predecir el siguiente token.

## Self-attention — intuición

Para cada palabra, el modelo calcula **cuánto debe "prestarle atención"** a cada otra palabra de la secuencia para representarla correctamente.

> "El banco cerró temprano porque el gerente **se enfermó**."

El modelo necesita relacionar "se enfermó" con "el gerente", no con "el banco" — eso es lo que resuelve la atención.

<img src="https://upload.wikimedia.org/wikipedia/commons/e/e4/Self-Attention_%28Scaled_dot-product_Attention%29.png" width="380"/>

<sub>Fuente: Wikimedia Commons, CC BY 4.0</sub>

## Self-attention — la ecuación

Cada token se proyecta en **Query (Q)**, **Key (K)** y **Value (V)**, con matrices de pesos aprendidas:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V
$$

- $QK^T$: producto punto Q·K → cuánto se relaciona cada token con cada otro.
- $\sqrt{d_k}$: escala para que el producto punto no sature el softmax.
- $\text{softmax}(\cdot)$: convierte los scores en pesos de atención (suman 1).
- Resultado: promedio ponderado de los $V$, según cuánto le "importa" a cada token.

**Multi-head attention**: varias atenciones en paralelo ("cabezas"), cada una capturando un tipo de relación distinto, después concatenadas.

## Tokenización BPE

Los modelos no procesan palabras, procesan **tokens**.

- **BPE (Byte Pair Encoding)**: fragmenta el texto en subpalabras frecuentes, balanceando vocabulario chico vs. secuencias largas.
- Consecuencias prácticas:
  - Palabras raras o en otros idiomas consumen más tokens.
  - El costo y los límites de contexto se miden en tokens, no en caracteres ni palabras.

## BPE — el algoritmo

1. Diccionario de palabras del corpus **con su frecuencia**, cada palabra como caracteres + símbolo de fin de palabra.
2. Contar la frecuencia de cada par de símbolos adyacentes, **ponderada por la frecuencia de la palabra**.
3. Fusionar el par más frecuente en un token nuevo, en todas las palabras donde aparece.
4. Repetir un número fijo de iteraciones (define el vocabulario final, típicamente 30k-100k tokens).

El paso 2 es clave: sin ponderar por frecuencia de palabra, "el par más frecuente" no está bien definido.

## BPE — ejemplo verificado

Corpus de 5 palabras con frecuencia (`_` = fin de palabra), ejemplo clásico de Sennrich et al. (2015):

| Palabra | Frecuencia |
|---|---|
| low | 5 |
| lowest | 2 |
| newer | 6 |
| wider | 3 |
| new | 2 |

```
1: "e r"  -> "er"    (newer×6 + wider×3 = 9)
2: "er _" -> "er_"   (9)
3: "n e"  -> "ne"    (newer×6 + new×2 = 8)
4: "ne w" -> "new"   (8)
5: "l o"  -> "lo"    (low×5 + lowest×2 = 7)
6: "lo w" -> "low"   (7)
```

## BPE — vista general del proceso

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/bpe-overview.webp" width="520"/>

<sub>Fuente: Sebastian Raschka, "Implementing A Byte Pair Encoding (BPE) Tokenizer From Scratch".</sub>

Consecuencia práctica: palabras frecuentes en el idioma/dominio de entrenamiento terminan en 1 token; palabras raras, nombres propios o texto en otros idiomas se fragmentan más → **consumen más tokens** para el mismo contenido, impactando costo y ventana de contexto.

## Tokens y ventana de contexto

- **Ventana de contexto**: cantidad máxima de tokens (entrada + salida) que el modelo puede procesar en una sola llamada.
- Todo lo que no entra en la ventana, el modelo no lo "ve".
- Implicancia directa para RAG y agentes: hay que decidir qué información entra y qué se descarta o resume.

## Generación probabilística

En cada paso, el modelo obtiene una distribución de probabilidad sobre el próximo token y **muestrea** de ella (no siempre elige el más probable).

- **Temperatura**: controla qué tan "conservador" o "creativo" es el muestreo.
  - Temperatura baja (~0) → respuestas más determinísticas, repetibles.
  - Temperatura alta → más variedad, más riesgo de desvíos.
- Otros parámetros: `top_p`, `max_tokens`, `stop sequences`.

<img src="https://upload.wikimedia.org/wikipedia/commons/4/4c/Fair_dice_probability_distribution.svg" width="320"/>

<sub>Ejemplo genérico de distribución de probabilidad discreta. Fuente: Oleg Alexandrov, Wikimedia Commons, dominio público</sub>

## Temperatura — la ecuación

$$
P(\text{token}_i) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}
$$

- $T \to 0$: distribución casi toda concentrada en el logit más alto → greedy decoding.
- $T = 1$: distribución "tal cual" la aprendió el modelo.
- $T > 1$: distribución más plana → más variedad, más riesgo de incoherencia.

<img src="https://huggingface.co/blog/assets/02_how-to-generate/sampling_search_with_temp.png" width="480"/>

<sub>Efecto de bajar la temperatura: la distribución se "afila" alrededor del token más probable. Fuente: Patrick von Platen, "How to generate text", Hugging Face Blog.</sub>

**Top-p (nucleus sampling)**: recorta a los tokens cuya probabilidad acumulada supera un umbral $p$ (ej. 0.9), y muestrea solo entre esos.

## Generación vs. embeddings vs. reranking

| Tarea | Qué produce | Para qué se usa |
|---|---|---|
| **Generación** | Texto nuevo | Responder, redactar, razonar |
| **Embeddings** | Vector numérico que representa significado | Buscar por similitud semántica |
| **Reranking** | Score de relevancia entre query y documento | Reordenar resultados de una búsqueda |

Un sistema RAG típico usa **los tres**, en distintas etapas.

<img src="https://upload.wikimedia.org/wikipedia/commons/9/94/T-SNE_visualisation_of_word_embeddings_generated_using_19th_century_literature.png" width="340"/>

<sub>Espacio de embeddings proyectado en 2D (t-SNE). Fuente: Siobhán Grayson, Wikimedia Commons, CC BY-SA 4.0</sub>

## Embeddings — similitud coseno

Un embedding es un vector $\vec{v} \in \mathbb{R}^d$. La similitud entre dos textos:

$$
\text{sim}(\vec{a}, \vec{b}) = \frac{\vec{a} \cdot \vec{b}}{\lVert \vec{a} \rVert \, \lVert \vec{b} \rVert}
$$

Cercano a 1 → vectores casi en la misma dirección → textos semánticamente similares. Cercano a 0 → sin relación. Base de la recuperación semántica que se construye desde la Clase 4-5 — hoy alcanza con la intuición: "vectores parecidos = significado parecido".

## Reranking — bi-encoder vs. cross-encoder

- **Bi-encoder** (modelo de embeddings, ej. Cohere Embed 4): codifica query y documento **por separado**, compara con similitud coseno. Rápido — se precalcula una vez — pero menos preciso.
- **Cross-encoder** (reranker): codifica query y documento **juntos**, devuelve un score de relevancia. Muy preciso, pero no escala — solo sirve para reordenar pocos candidatos.

Patrón típico (Clase 8): **recuperar con bi-encoder (todo el corpus) → re-ordenar el top-k con cross-encoder (pocos candidatos)**.

<img src="https://weaviate.io/assets/images/weaviate-pipeline-long-9bc0be65ab51ae7837a2ea37997f08fd.png" width="460"/>

<sub>Fuente: Weaviate Blog, "Using Cross-Encoders as reranker in multistage vector search".</sub>

## Modelos propietarios vs. abiertos

**Propietarios** (Claude, GPT, Gemini, ...)
- Mejor performance de punta, sin infraestructura propia.
- Se paga por uso, dependencia del proveedor.

**Abiertos** (Llama, Mistral, ...)
- Se pueden auto-hospedar → control total de datos y costos fijos.
- Requieren infraestructura y expertise para servir con buena latencia.

## Costo, latencia y privacidad

- **Costo**: se cobra por tokens de entrada y salida — el diseño del prompt y del contexto impacta directamente el costo.
- **Latencia**: modelos más grandes suelen ser más lentos; importa el uso (chat interactivo vs. batch).
- **Privacidad**: qué datos viajan al proveedor, dónde se procesan, qué garantías contractuales existen (relevante para datos sensibles).

## Criterios de selección de modelos

No existe "el mejor modelo" en abstracto. Se elige según:

1. Calidad para la tarea específica (no benchmarks genéricos).
2. Costo por token y volumen esperado.
3. Latencia tolerable por el caso de uso.
4. Requisitos de privacidad/compliance.
5. Disponibilidad de la función necesaria (tool calling, structured outputs, embeddings, etc.).

## Specs reales — ejemplo de referencia (AWS Bedrock)

El curso es agnóstico de proveedor (Clase 3: "Abstracción de proveedores"). Estas specs son el ejemplo de los laboratorios de la cátedra — cualquiera de OpenAI, Google o Anthropic directo funciona igual, solo cambia el model ID.

### Claude Sonnet 4.6 (Anthropic)

| Spec | Valor |
|---|---|
| Ventana de contexto | 1.000.000 tokens |
| Máx. tokens de salida | 64.000 tokens |
| Modalidades | Entrada: texto, imagen · Salida: texto |
| Tool calling / structured outputs | Soportado |
| Model ID (Bedrock) | `global.anthropic.claude-sonnet-4-6` |

## Specs reales — Cohere Embed 4

| Spec | Valor |
|---|---|
| Contexto máximo por documento | ~128.000 tokens |
| Dimensiones configurables | 256 / 512 / 1024 / 1536 |
| Tipos de entrada | Texto, imagen, texto+imagen intercalado |
| Tipos de embedding | float, int8, uint8, binary, ubinary |
| Model ID (Bedrock) | `cohere.embed-v4:0` |

**Nota sobre costo**: Bedrock cobra por token, con tarifas que cambian — revisar el precio vigente en la [página de pricing de Bedrock](https://aws.amazon.com/bedrock/pricing/) en vez de memorizar un número que puede quedar desactualizado.

## Trabajo Práctico Final Integrador

A lo largo de las 12 clases vas a construir, junto con tus compañeros, **un único sistema**:

- Pipeline **RAG** sobre un corpus de documentación técnica (provisto por la cátedra).
- Un **flujo con agentes/herramientas** que puede invocar acciones externas.
- El sistema **desplegado** vía API + demo.

Cada clase suma un checkpoint. Ver la consigna completa en `proyecto-integrador/README.md`.

## Stack técnico del curso

- El curso es **agnóstico de proveedor**: Anthropic, OpenAI, Google, AWS Bedrock o modelos abiertos locales (Ollama).
- Desde la Clase 3 se programa contra una **interfaz propia** — cambiar de proveedor es configuración, no reescribir código.
- Para practicar hoy: la cátedra provee acceso a AWS Bedrock (Claude Sonnet 4.6 + Cohere Embed 4), pero cualquier credencial propia de otro proveedor sirve igual.
- Credenciales solo por variables de entorno (`.env`), nunca hardcodeadas.

## Actividad de hoy

**Matriz de selección de modelos**: comparar 3-4 modelos (al menos uno propietario y uno abierto) según calidad, costo, latencia, privacidad y disponibilidad de funciones, aplicado al caso del TP integrador.

Ver `actividad.md` para la consigna completa.

## Lecturas introductorias (contenido asincrónico)

Para quien quiera ir más al fondo, antes o después de la clase:

- Vaswani et al., *Attention Is All You Need* (2017) — paper original de la arquitectura Transformer.
- Documentación de Anthropic sobre parámetros de generación (`temperature`, `top_p`, `max_tokens`).
- AWS Bedrock: [Cohere Embed v4](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-embed-v4.html) y [ficha de Claude Sonnet 4.6](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-sonnet-4-6.html).

## Cierre

- Un LLM predice tokens, no "sabe" — de ahí vienen sus capacidades y sus límites.
- La elección de modelo es una decisión de ingeniería, no solo de calidad.
- Próxima clase: pasamos del modelo a una aplicación real — APIs, prompting y configuración del entorno.